# Ch 5. TinyStories와 합성 데이터

**1M 모델이 일관된 동화를 짓는 충격적 결과의 데이터 — TinyStories와 합성 데이터의 시대를 손으로 확인한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part2/ch05_tinystories.ipynb)

In [ ]:
# !pip install -q datasets anthropic

import torch
print(f"PyTorch: {torch.__version__}")

try:
    import datasets
    print(f"datasets: {datasets.__version__}")
except ImportError:
    print("datasets not installed — run: pip install datasets")

## 최소 예제 — TinyStories 한 번 들여다보기

TinyStories의 핵심 아이디어:
1. **3~4세 어린이 어휘**(약 1,500개)로 동화를 GPT-3.5가 합성
2. **2.4M개** 합성 동화 (약 200M 토큰)
3. 이 데이터로 1M~33M 모델 훈련

**관찰**:
- 어휘가 **단순** — "happy", "big", "red", "sky" 같은 기초 단어
- 한 동화 길이 **400~1,500자** (약 100~400 토큰)
- 구조 **반복** — "Once upon a time + 인물 + 사건 + 해결"

이 단순함이 **1M 모델로도 학습 가능한 이유**.

In [ ]:
# peek_tinystories.py
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
for i, row in enumerate(ds):
    if i >= 3: break
    print(f"--- Story {i} ({len(row['text'])} chars) ---")
    print(row["text"][:300], "...\n")

In [ ]:
# TinyStories 길이 분포 확인 (100개 샘플)
from datasets import load_dataset
import statistics

ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
lengths = []
for i, row in enumerate(ds):
    if i >= 100: break
    lengths.append(len(row["text"]))

print(f"샘플 100개 길이 통계 (글자 수):")
print(f"  평균: {statistics.mean(lengths):.0f}")
print(f"  중앙값: {statistics.median(lengths):.0f}")
print(f"  표준편차: {statistics.stdev(lengths):.0f}")
print(f"  최소: {min(lengths)}")
print(f"  최대: {max(lengths)}")

## 실전 — 한국어 합성 데이터 만들기

본 책 캡스톤용 — TinyStories-KO 5,000개 동화를 LLM으로 합성.

**핵심**: 인물·키워드를 매번 바꿔 **다양성** 확보. 같은 프롬프트만 5,000번 돌리면 비슷한 동화만 나옴.

**비용**: Haiku 모델 기준 5,000건 × 약 500 토큰 ≈ 2.5M 토큰 → 약 $1~2

**라이선스**:
- 본인이 합성 → Anthropic ToS는 모델 학습 데이터로 사용 가능
- TinyStories 영어판 섞을 때 → CDLA-Sharing 2.0

In [ ]:
# synth_prompt.py — 프롬프트 설계
PROMPT = """3~5세 어린이가 듣는 한국어 동화 한 편을 만들어줘.

규칙:
- 길이: 200~400자
- 어휘: 매우 단순. 어려운 한자어 X.
- 구조: 인물 등장 → 작은 사건 → 해결
- 등장인물: {character}
- 키워드: {keyword1}, {keyword2}

따뜻한 톤으로. 동화 본문만 출력 (제목·해설 없이).
"""

characters = ["토끼 토토", "곰 두두", "할머니", "고양이 미미", "아빠"]
keywords_pool = ["당근", "비", "달", "친구", "엄마", "꽃", "신발", "나무", "새", "강아지"]

import random
char = random.choice(characters)
kws = random.sample(keywords_pool, 2)
print("샘플 프롬프트:")
print(PROMPT.format(character=char, keyword1=kws[0], keyword2=kws[1]))

In [ ]:
# synth_run.py — 소규모 테스트 (10개만)
# Anthropic API 키가 있어야 동작. 없으면 건너뛰기.
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY 환경변수가 없습니다.")
    print("Colab에서: from google.colab import userdata; os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')")
else:
    import anthropic, json, random
    client = anthropic.Anthropic()

    out = []
    n_generate = 10  # 테스트는 10개만 (전체는 5000)
    for i in range(n_generate):
        char = random.choice(characters)
        kws = random.sample(keywords_pool, 2)
        msg = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=600,
            messages=[{"role": "user", "content":
                PROMPT.format(character=char, keyword1=kws[0], keyword2=kws[1])}]
        )
        out.append({"text": msg.content[0].text})
        if i % 5 == 0: print(f"  {i}/{n_generate}")

    # JSONL 저장
    with open("tinystories_ko_test.jsonl", "w") as f:
        for row in out:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"\n생성 완료: {len(out)}개 동화")
    print("\n첫 번째 동화:")
    print(out[0]["text"])

In [ ]:
# filter.py — 품질 필터
def passes(text):
    if len(text) < 150 or len(text) > 600: return False       # 길이
    if text.count("\n\n") > 3: return False                   # 단락 많음 (이상)
    if any(w in text for w in ["GPT", "AI", "Claude", "버전"]): return False  # 메타 누출
    if text.count("같은") > 5: return False                    # 단조 반복
    return True

# 테스트
test_texts = [
    "토끼 토토는 당근을 좋아했어요. 어느 날 비가 내렸어요. 토토는 우산을 들고 나갔어요. 빗속에서 친구를 만났어요. 둘이 함께 집으로 돌아왔어요. 그날 밤 토토는 행복하게 잠들었어요.",
    "Claude AI가 생성한 버전입니다.",  # 메타 누출 — 필터 통과 안 함
    "짧다",  # 너무 짧음
]
for t in test_texts:
    print(f"'{t[:30]}...' -> {'통과' if passes(t) else '제거'}")

## 연습문제

1. `roneneldan/TinyStories`에서 100개 동화를 다운로드해 길이 분포(글자 수) 히스토그램을 그려라. 평균·중앙값·표준편차.
2. §5.1의 프롬프트로 한국 동화 10개를 직접 합성해보고, 다음 두 가지를 측정:
   - 평균 길이 (한국어 글자 수)
   - 중복률 (Jaccard similarity > 0.5인 쌍의 비율)
3. 같은 프롬프트로 **temperature=0.3** vs **temperature=1.2**로 각 10개 합성. 다양성 차이는?
4. 합성 데이터에 "GPT가" 또는 "이 이야기는" 같은 메타 문장이 몇 % 나오는가? 100개 샘플 기준.
5. **(생각해볼 것)** 본인 도메인 (예: 콜센터 대화, 레시피, 기술 문서)에 TinyStories 정신을 적용하면 어떤 합성 프롬프트가 좋겠는가? 인물·키워드·구조 3 요소로 설계.

In [ ]:
# 연습문제 1 — TinyStories 100개 길이 분포
from datasets import load_dataset
import statistics

ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
lengths = []
for i, row in enumerate(ds):
    if i >= 100: break
    lengths.append(len(row["text"]))

# 히스토그램 (텍스트 버전)
print(f"길이 분포 (100개 샘플):")
print(f"  평균: {statistics.mean(lengths):.0f}")
print(f"  중앙값: {statistics.median(lengths):.0f}")
print(f"  표준편차: {statistics.stdev(lengths):.0f}")

# 버킷별 카운트
buckets = [0, 300, 500, 700, 1000, 1500, 9999]
labels = ["<300", "300-500", "500-700", "700-1000", "1000-1500", "1500+"]
counts = [0] * len(labels)
for l in lengths:
    for j in range(len(buckets)-1):
        if buckets[j] <= l < buckets[j+1]:
            counts[j] += 1
            break

print("\n히스토그램:")
for label, cnt in zip(labels, counts):
    bar = "#" * cnt
    print(f"  {label:10s} | {bar} ({cnt})")

In [ ]:
# 연습문제 2 — 합성 동화 Jaccard 유사도 측정
def jaccard(a, b, n=3):
    """n-gram 기반 Jaccard 유사도."""
    def ngrams(text, n):
        return set(text[i:i+n] for i in range(len(text)-n+1))
    sa, sb = ngrams(a, n), ngrams(b, n)
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

# 테스트 문장들로 시연
s1 = "토끼 토토는 당근을 좋아했어요. 어느 날 비가 내렸어요."
s2 = "토끼 토토는 당근을 좋아했어요. 어느 날 눈이 내렸어요."
s3 = "곰 두두는 달을 바라보았어요. 달이 참 예뻤어요."

print(f"s1 vs s2 (비슷): {jaccard(s1, s2):.3f}")
print(f"s1 vs s3 (다름): {jaccard(s1, s3):.3f}")
print("\n합성 동화가 있으면 tinystories_ko_test.jsonl을 읽어 중복률 계산:")
print("  import json")
print("  with open('tinystories_ko_test.jsonl') as f:")
print("      texts = [json.loads(l)['text'] for l in f]")

In [ ]:
# 연습문제 3 — temperature 다양성 비교 (API 없이 프록시 실험)
# SmolLM2로 temperature 효과 시연
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

name = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32)

prompt = "Once upon a time, there was a little"
ids = tok(prompt, return_tensors="pt").input_ids

print("temperature=0.3 (낮음 — 반복 많음):")
for _ in range(3):
    out = model.generate(ids, max_new_tokens=20, do_sample=True, temperature=0.3)
    print(" ", tok.decode(out[0], skip_special_tokens=True))

print("\ntemperature=1.2 (높음 — 다양성 많음):")
for _ in range(3):
    out = model.generate(ids, max_new_tokens=20, do_sample=True, temperature=1.2)
    print(" ", tok.decode(out[0], skip_special_tokens=True))

In [ ]:
# 연습문제 4 — 메타 누출 패턴 탐지
meta_patterns = ["GPT가", "GPT-", "이 이야기는", "Claude", "AI가", "언어 모델", "인공지능"]

def detect_meta(text):
    return [p for p in meta_patterns if p in text]

# 테스트
test_cases = [
    "토끼 토토는 숲 속에서 살았어요.",
    "이 이야기는 GPT가 만든 동화입니다.",
    "Claude AI가 생성한 내용입니다.",
]
print("메타 누출 탐지 테스트:")
for t in test_cases:
    found = detect_meta(t)
    print(f"  '{t[:30]}...' -> {found if found else '클린'}")

In [ ]:
# 연습문제 5 — 본인 도메인 합성 프롬프트 설계
# 예시: 콜센터 대화 도메인
MY_DOMAIN_PROMPT = """고객 서비스 대화를 한 건 작성해줘.

규칙:
- 도메인: 통신사 고객센터
- 고객 요구사항: {issue}
- 상담사 이름: {agent_name}
- 길이: 5~8 대화 턴
- 어조: 친절하고 전문적
- 형식: 고객: / 상담사: 형식
"""

issues = ["요금 문의", "데이터 초과", "번호 변경", "해지 요청"]
agents = ["김민지", "이준호", "박서연"]

import random
print("샘플 도메인 프롬프트:")
print(MY_DOMAIN_PROMPT.format(
    issue=random.choice(issues),
    agent_name=random.choice(agents)
))